# ▶ EJECUTAR TODO — VERIFAKE (TFM)

**Un solo cuaderno**, en Colab o Kaggle, guardando siempre los datos en **tu Google
Drive**. Encadena el pipeline completo y **lanza la app con su URL pública**.

### Cómo usarlo
1. GPU: Colab → *Entorno de ejecución → T4 GPU*. Kaggle → *Settings → Accelerator → GPU*
   y *Settings → Internet → On*.
2. En la celda 1, edita `REPO_URL`.
3. Ejecuta todo.

### Persistencia en Drive
- **Colab:** monta Drive y escribe directo en él. Nada que configurar.
- **Kaggle:** sincroniza con rclone. Configúralo UNA vez:
  1. En tu ordenador: instala rclone y `rclone config` → remoto **`gdrive`**, tipo
     **`drive`**, login de Google.
  2. `rclone config file` → abre el fichero y copia el bloque `[gdrive] ... token=...`.
  3. Kaggle → *Add-ons → Secrets* → secreto **`rclone_conf`** con ese contenido.
  El notebook extrae el token y reconstruye la config, así que aunque Kaggle pierda
  los saltos de línea, funcionará.


## 1. Preparar el entorno (detecta Colab o Kaggle)

In [ ]:
import os, sys
from pathlib import Path

IN_KAGGLE = Path("/kaggle/working").exists()
IN_COLAB = Path("/content").exists() and not IN_KAGGLE

REPO_URL = "https://github.com/pablo-Tesoro/tfm-deepfake-detection.git"
DRIVE_DIR = "TFM_Deepfake"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKSPACE = f"/content/drive/MyDrive/{DRIVE_DIR}"
    PROJECT_ROOT = "/content/TFM_Deepfake_Detection"

os.environ["TFM_WORKSPACE"] = WORKSPACE
os.environ["TFM_PROJECT_ROOT"] = PROJECT_ROOT
Path(WORKSPACE).mkdir(parents=True, exist_ok=True)

if not Path(PROJECT_ROOT).exists():
    !git clone {REPO_URL} {PROJECT_ROOT}
else:
    !cd {PROJECT_ROOT} && git pull -q

!pip install -q timm grad-cam gradio pyyaml tqdm seaborn
!pip install -q --no-deps facenet-pytorch

print(f"Entorno: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'local'} | Workspace: {WORKSPACE}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Entorno: Colab | Workspace: /content/drive/MyDrive/TFM_Deepfake


## 2. Conectar con Drive y traer los datos existentes

- **Colab:** nada que hacer.
- **Kaggle:** reconstruye la config de rclone desde el secreto, prueba la conexión y
  descarga de Drive lo que ya tengas.

## 3. Ejecutar todo y lanzar la app

Encadena: splits → (descarga opcional) → **vídeo→embeddings fusionados** →
entrenamiento → evaluación → **experimentos** (curva de aprendizaje,
EfficientNet vs ResNet, métricas por método) → app.

> La comparativa de backbones añade, **solo la primera vez**, una pasada
> completa vídeo→embedding para ResNet (luego queda cacheada). Ponla a `None`
> si quieres saltártela.

In [ ]:
sys.path.insert(0, os.environ["TFM_PROJECT_ROOT"])
from run_all import run_pipeline

run_pipeline(
    download=False,               # los vídeos ya están en el Drive
    retrain=True,                 # reentrenar con el dataset completo
    make_figs=True,               # figuras básicas para la memoria
    experiments=True,             # los 3 experimentos avanzados
    compare_backbone="resnet50",  # activa la comparativa EfficientNet vs ResNet
    launch=False,                 # app con enlace público
)


[1/7]  Splits oficiales
Splits oficiales: ya presentes.

[2/7]  Inventario de vídeos
5000 vídeos (1000 reales / 4000 fakes).

[3/7]  Vídeo -> embeddings (fusionado, MTCNN compartido para 2 backbones)


model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

Fusionado | backbones=['efficientnet_b0', 'resnet50'] | device=cuda | pendientes=2368/5000


Video -> embedding: 100%|██████████| 5000/5000 [4:48:50<00:00,  3.47s/it]


  efficientnet_b0: 5000 vídeos con embeddings
  resnet50: 5000 vídeos con embeddings

[4/7]  Entrenamiento (baseline + híbrido)
Reparto: split OFICIAL de FaceForensics++.
  train: 3600 vídeos | fakes=2880
  val  :  700 vídeos | fakes=560
  test :  700 vídeos | fakes=560
Entrenando baseline...
Entrenando híbrido CNN+LSTM...

[5/7]  Evaluación y figuras para la memoria

Métricas en test:
                   accuracy  precision  recall      f1     auc
Baseline            0.7300     0.9187  0.7268  0.8116  0.8140
Híbrido CNN+LSTM    0.8086     0.8944  0.8625  0.8782  0.8346
Figuras guardadas en /content/drive/MyDrive/TFM_Deepfake/reports/figures

[6/7]  Experimentos avanzados

-- Curva de aprendizaje (AUC vs nº de vídeos) --
Reparto: split OFICIAL de FaceForensics++.
  n_train= 720 -> AUC=0.726 | F1=0.871
  n_train=1440 -> AUC=0.777 | F1=0.853
  n_train=2160 -> AUC=0.786 | F1=0.793
  n_train=2880 -> AUC=0.813 | F1=0.802
  n_train=3600 -> AUC=0.825 | F1=0.870

-- Backbones: efficientnet_b0 v

## 4. Ejecutar solo la APP

In [ ]:
import os, sys
sys.path.insert(0, os.environ["TFM_PROJECT_ROOT"])
from app.app import build_demo
build_demo().launch(share=True)

/content/TFM_Deepfake_Detection/app/app.py:348: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme, css=CUSTOM_CSS, title="VERIFAKE") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://78ed4cd207213a2afe.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
